# [CS303]Practice 6 - BTS & Practice 7 - Improving BTS
In Practice 6 and 7, we will apply the BTS algorithm and the Improving BTS to solve the N-Queens problem.

In the following code, only two of the function usages are essential. One is `is_valid(self, state)`, which checks if the current state is legal; the other is `is_satisfy(self, state)`, which is to checks if the current board meets the win condition. 
The type of state is [], which stores the tuples(a, b), representing the positions of the queens in it.

In test block, for `Solver`,  `n` indicates the size of the test while `method` indicates which bts function will be used(`bts` or `improving_bts`); for method `solve`, `render` indicates whether to use a graphical interface representation.

## Practice 6 - BTS
You are required to implement complete the function `bts()`.
You can only use iterative way, not recursive. Using recursion will incur a **20% penalty**. You may add any function you want. 
**Remember to test the case where N=6**.
#### **Practice 6 Deadline and Checking Policy**

| Lab Group     | Full Credit (100%)           | Late Check (50%)        | After the 50% Deadline |
| ------------- | ---------------------------- | ----------------------- | ---------------------- |
| **Wednesday** | **Oct. 21** or **Oct. 28**   | **Nov. 04**             | 0%                     |
| **Thursday**  | **Oct. 22** or **Oct. 29**   | **Nov. 05**             | 0%                     |
| **Friday**    | **Oct. 23** or **Oct. 30**   | **Nov. 06**             | 0%                     |

## Practice 7 - Improving BTS
You are required to complete the function `improving_bts()`.
For `improving_bts()`, you can use one or more of the following strategies: Minimum Remaining Value, Least constraining value, and Forward checking. You should have a good performance (within 2s) **when N = 16 without GUI**. 
#### **Practice 7 Deadline and Checking Policy**

| Lab Group     | Full Credit (100%)           | Late Check (50%)        | After the 50% Deadline |
| ------------- | ---------------------------- | ----------------------- | ---------------------- |
| **Wednesday** | **Oct. 28** or **Nov. 04**   | **Nov. 11**             | 0%                     |
| **Thursday**  | **Oct. 29** or **Nov. 05**   | **Nov. 12**             | 0%                     |
| **Friday**    | **Oct. 30** or **Nov. 06**   | **Nov. 13**             | 0%                     |

## Evaluation Criteria

Your submission will be assessed on:

* Clear understanding of every line of your code (no blind copy-paste).
* Successful compilation of the program.
* Correctness of the program logic.
* Reasonable efficiency of the solution.



In [19]:
import numpy as np
import time

def my_range(start, end):
    if start <= end:
        return range(start, end + 1)
    else:
        return range(start, end - 1, -1)


class Problem:
    char_mapping = ('·', 'Q')

    def __init__(self, n=4):
        self.N = n

    def is_valid(self, state):
        """
        check whether the state satisfies the condition or not.
        : param state: list of points (in (row id, col id) tuple form)
        : return: bool value of valid or not
        """
        board = self.get_board(state)
        res = True
        for point in state:
            i, j = point
            condition1 = board[:, j].sum() <= 1
            condition2 = board[i, :].sum() <= 1
            condition3 = self.pos_slant_condition(board, point)
            condition4 = self.neg_slant_condition(board, point)
            res = res and condition1 and condition2 and condition3 and condition4
            if not res:
                break
        return res

    def is_satisfy(self, state):
        return self.is_valid(state) and len(state) == self.N

    def pos_slant_condition(self, board, point):
        i, j = point
        tmp = min(self.N - i - 1, j)
        start = (i + tmp, j - tmp)
        tmp = min(i, self.N - j - 1)
        end = (i - tmp,  j + tmp)
        rows = my_range(start[0], end[0])
        cols = my_range(start[1], end[1])
        return board[rows, cols].sum() <= 1

    def neg_slant_condition(self, board, point):
        i, j = point
        tmp = min(i, j)
        start = (i - tmp, j - tmp)
        tmp = min(self.N - i - 1, self.N - j - 1)
        end = (i + tmp, j + tmp)
        rows = my_range(start[0], end[0])
        cols = my_range(start[1], end[1])
        return board[rows, cols].sum() <= 1

    def get_board(self, state):
        board = np.zeros([self.N, self.N], dtype=int)
        for point in state:
            board[point] = 1
        return board

    def print_state(self, state):
        board = self.get_board(state)
        print('_' * (2 * self.N + 1))
        for row in board:
            for item in row:
                print(f'|{Problem.char_mapping[item]}', end='')
            print('|')
        print('-' * (2 * self.N + 1))

In [20]:
class Solver:
    def __init__(self, n, method):
        self.N = n
        self.problem = Problem(n)
        self.method = method

    def solve(self, render=True):
        if render:
            import pygame
            w, h = 90 * self.n + 10, 90 * self.N + 10
            screen = pygame.display.set_mode((w, h))
            screen.fill('white')
            action_generator = self.method(self.problem)
            clk = pygame.time.Clock()
            queen_img = pygame.image.load('./queen.png')
            while True:
                for event in pygame.event.get():
                    if event == pygame.QUIT:
                        exit()
                try:
                    actions = next(action_generator)
                    screen.fill('white')
                    for i in range(self.n + 1):
                        pygame.draw.rect(screen, 'black', (i * 90, 0, 10, h))
                        pygame.draw.rect(screen, 'black', (0, i * 90, w, 10))
                    for action in actions:
                        i, j = action
                        screen.blit(queen_img, (10 + 90 * j, 10 + 90 * i))
                    pygame.display.flip()
                except StopIteration:
                    pass
                clk.tick(5)
            pass
        else:
            start_time = time.time()
            for actions in self.method(self.problem):
                pass
            self.problem.print_state(actions)
            print(time.time() - start_time)

# BTS: Backtracking search

In [21]:
def bts(problem):
    action_stack = []
    while not problem.is_satisfy(action_stack):
        # TODO: Implement backtracking search logic here

        # rows are assigned in order: 0, 1, 2, ...
        # so the number of placed queens is the next row index
        # so current row index is the number of placed queens currently
        row = len(action_stack)
        col = 0

        # keep searching until we either place one queen or exhaust all columns
        # if we exhaust all columns, backtrack by removing the last placed queen
        while True:

            # try columns from left to right in the current row.
            while col < problem.N:

                # build a temporary state without changing action_stack,
                # by appending the candidate position to the current action stack
                candidate = action_stack + [(row, col)]

                # only commit the queen if the entire state is legal,
                # that is, the candidate state is valid according to the queen game's regulations
                if problem.is_valid(candidate):
                    action_stack.append((row, col))
                    break

                col += 1

            # if we have found a legal position for the current row,
            # all we have to do is to break out the inner while loop,
            # and continue to the next row.
            if col < problem.N:
                break

            # only if we have exhausted all columns for the current row,
            # but still fail to find a valid position,
            # can we reach this point,
            # which indicates that we need to backtrack by removing the last placed queen.

            # however, there is no last placed queen for us to remove,
            # cuz currently the action_stack is just empty,
            # it means we have reached the beginning of the search,
            # and still cannot find a valid solution,
            # ---> no solution at all !!!
            # we shall yield an empty list and return.
            if not action_stack:
                yield []
                return

            # if we reach this point,
            # if means that the action_stack is not empty,
            # thus there is a last placed queen for us to backtrack
            row, col = action_stack.pop()

            # this pair of row & col represents the last placed queen that we are backtracking from.
            # we will continue the search from this row, trying the next column after the one we just tried.
            # if we do not increment the col, we will meet the failure case again,
            # which is redundant and should be avoided.
            col += 1

        yield action_stack

# Improving BTS 
* Which variable should be assigned next?
* In what order should its values be tried?
* Can we detect inevitable failure early?

## Minimum Remaining Value
Choose the variable with the fewest legal values in its domain
## Least constraining value
Given a variable, choose the least constraining value: the one that rules out the fewest values in the remaining variables
## Forward checking
Keep track of remaining legal values for the unassigned variables. Terminate when any variable has no legal values.

In [22]:
def improving_bts(problem):
    action_stack = []
    while not problem.is_satisfy(action_stack):
        # TODO: Implement improving bts logic here

        # for a newly selected row,
        # it starts searching from column 0.
        # but after backtracking,
        # this becomes the previous column + 1.
        # we maintain this to avoid revisiting columns that have already been tried for this row.
        start_col = 0

        while True:

            # get the set of rows that are already used by the queens in the action_stack.
            # this helps us to avoid placing a queen in a row that is already occupied.
            # we will use this set to filter out invalid row choices when selecting the next row for a queen.
            used_rows = {r for r, c in action_stack}

            # initialize the domains dictionary to store the possible column values for each row.
            domains = {}

            # recompute the legal columns of every unassignd row.
            for row in range(problem.N):
                if row in used_rows:
                    continue

                # Initially, every column is a possible value
                legal_cols = set(range(problem.N))

                # remove positions attacked by each existing queen,
                # so we traverse through all existing queens in action_stack.
                for queen_row, queen_col in action_stack:
                    distance = abs(row - queen_row)

                    # note that, an existing queen attcks:
                    # 1. its own column
                    # 2. one diagonal to the left
                    # 3. one diagonal to the right

                    # here, different_update is used to remove the columns below,
                    # which are attacked by the existing queen,
                    # by our rule of the game.
                    legal_cols.difference_update({
                        queen_col,
                        queen_col - distance,
                        queen_col + distance,
                    })

                # after removing all attacked columns,
                # the remaining legal columns are stored in the domains dictionary,
                # which are all posible column values for that row.
                domains[row] = legal_cols

            # after reaching this point,
            # we have computed the domains for all unassigned rows,
            # which represent the possible column values for each row.
            
            # the intuition is,
            # the row with the smallest domain will be considered first,
            # this helps to reduce the branching factor and improve the efficiency of the search.
            row = min(domains, key=lambda r: (len(domains[r]), r))
            # lambda is used to define a custom key function for the min function,
            # which selects the row with the smallest domain size, and if there are ties,
            # it selects the row with the smallest index.

            # now we have selected the row to assign a queen to next,
            # and we will try to assign a column from its domain.
            # only consider columns that are greater than or equal to start_col,
            # because we want to avoid revisiting columns that have already been tried for this row.
            candidates = [
                col for col in sorted(domains[row])
                if col >= start_col
            ]

            # commit the first candidate column to the action stack.
            # this is ensured to be a valid column for the current row,
            # as it is part of the domain and meets the start_col constraint.
            if candidates:
                action_stack.append((row, candidates[0]))
                break

            # if we reach this point,
            # it means that no candidate is available for the current row,
            # and we need to backtrack to try a different column for the previous row.
            # If every earlier decision has also failed, the problem has no solution.
            # Stop before trying to pop from an empty stack.
            if not action_stack:
                yield []
                return

            previous_row, previous_col = action_stack.pop()

            # after popping the previous action,
            # we will continue the loop to try a different column for the previous row.
            start_col = previous_col + 1
            # this ensures that we do not revisit the column that was just tried for the previous row.
        
        yield action_stack  

## test block

In [23]:
Solver(n=6, method=bts).solve(render=False)

_____________
|·|Q|·|·|·|·|
|·|·|·|Q|·|·|
|·|·|·|·|·|Q|
|Q|·|·|·|·|·|
|·|·|Q|·|·|·|
|·|·|·|·|Q|·|
-------------
0.00852513313293457


In [24]:
Solver(n=16, method=improving_bts).solve(render=False)

_________________________________
|Q|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|
|·|·|Q|·|·|·|·|·|·|·|·|·|·|·|·|·|
|·|·|·|·|Q|·|·|·|·|·|·|·|·|·|·|·|
|·|·|·|·|·|·|·|·|·|·|·|·|Q|·|·|·|
|·|·|·|·|·|·|·|·|·|·|Q|·|·|·|·|·|
|·|·|·|Q|·|·|·|·|·|·|·|·|·|·|·|·|
|·|·|·|·|·|·|·|·|·|·|·|·|·|·|Q|·|
|·|·|·|·|·|·|Q|·|·|·|·|·|·|·|·|·|
|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|Q|
|·|·|·|·|·|·|·|·|·|·|·|·|·|Q|·|·|
|·|Q|·|·|·|·|·|·|·|·|·|·|·|·|·|·|
|·|·|·|·|·|·|·|Q|·|·|·|·|·|·|·|·|
|·|·|·|·|·|Q|·|·|·|·|·|·|·|·|·|·|
|·|·|·|·|·|·|·|·|Q|·|·|·|·|·|·|·|
|·|·|·|·|·|·|·|·|·|·|·|Q|·|·|·|·|
|·|·|·|·|·|·|·|·|·|Q|·|·|·|·|·|·|
---------------------------------
0.007961034774780273
